## Landslide event mapping using Sentinel-2 data

Map the extent of landslides using Sentinel-2 data.

Generate a pre event mosaic and use the first available post event image. 
Calculate the NDVI for both and generate a change image using an NDVI threshold and the formula (pre-image * 10) + post-image to generate the following classes for each image:

- 00 indicates no change
- 01 change
- 10 is not reasonable because it means that pre-event is true and post not true and can be disregarded
- 11 both images are true --> in both images NDVI is below the threshold and no change happened

In [1]:
import openeo
from openeo.processes import quantiles
from pathlib import Path
from openeo.processes import array_element, quantiles, median
import leafmap
from shapely.geometry import shape
from folium.plugins import Draw

In [2]:
# define properties for the outputs
out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/blatten")
mkdir = out_dir.mkdir(parents=True, exist_ok=True)

event = "Blatten_2025_mosaics"

### 1) Define time frame and extent

In [3]:
# 1 . Define the time periods for pre- and post event date
# blatten

PRE_DATE  = ("2024-05-01", "2024-06-30")   
POST_DATE = ("2025-05-28", "2025-05-31")   

# Pre and post date merano
#PRE_DATE  = ("2026-06-01", "2026-06-28")   
#POST_DATE = ("2026-06-29", "2026-07-06")  

# vals
#PRE_DATE  = ("2026-07-01", "2026-07-30")   
#POST_DATE = ("2026-08-08", "2026-08-15")   

# Ecrins
#PRE_DATE  = ("2024-06-01", "2024-06-20")   
#POST_DATE = ("2024-06-25", "2024-07-05")

# Nepal
#PRE_DATE  = ("2026-07-01", "2026-08-20")   
#POST_DATE = ("2026-08-26", "2026-09-10")

In [4]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [5]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 7.789135, 'south': 46.384833, 'east': 7.867413, 'north': 46.427091}


### 2) Authentificate and load the pre and post Sentinel-2 cube

In [11]:
# Remove connection to the backend and remove the refresh token 
from openeo.rest.auth.config import RefreshTokenStore
RefreshTokenStore().remove()

In [12]:
# load sentinel-2 data after the flood event
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=INZX-KPTZ 📋 to authenticate.

✅ Authorized successfully

Authenticated using device code flow.
Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [13]:
BANDS   = ["B08", "B04"]         
BAD_SCL = [1, 3, 7, 8, 9, 10]

TIMESPAN = {"PRE": PRE_DATE, "POST": POST_DATE}   # each a (start, end) pair


def load_s2(temporal_extent):
    bands = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=20,
    ).resample_spatial(resolution=10, method="bilinear")

    scl = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=["SCL"],
        max_cloud_cover=50,
    ).resample_spatial(resolution=10, method="near")   # categorical -> nearest

    return bands, scl


def cloud_mask(scl):
    s = scl.band("SCL")
    m = s == BAD_SCL[0]
    for v in BAD_SCL[1:]:
        m = m | (s == v)
    return m


def masked_cube(temporal_extent):
    bands, scl = load_s2(temporal_extent)
    return bands.mask(cloud_mask(scl))


def composite_pre(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: array_element(quantiles(data=x, probabilities=[0.25]), index=0),
    )


def composite_post(temporal_extent):
    return masked_cube(temporal_extent).reduce_dimension(
        dimension="t",
        reducer=lambda x: median(x),
    )


def ndvi(mosaic):
    """NDVI = (NIR - Red) / (NIR + Red) = (B08 - B04) / (B08 + B04)"""
    def _ndvi(b):
        nir = b[0]   # B08
        red = b[1]   # B04
        return (nir - red) / (nir + red)
    return mosaic.reduce_dimension(dimension="bands", reducer=_ndvi)


COMPOSITES = {"PRE": composite_pre, "POST": composite_post}

# --- PRE and POST NDVI as batch jobs ---
jobs = {}
for name, extent in TIMESPAN.items():
    print(f"Starting {name}...")
    result = ndvi(COMPOSITES[name](extent)).save_result(format="GTiff")
    job = result.create_job(title=f"S2_NDVI_{name}")
    job.start()
    jobs[name] = job
    print(f"{name} started: {job.job_id}")


Starting PRE...
PRE started: j-2610060923234806a7f011f86630f163
Starting POST...


Failed to parse API error response: [429] 'Too Many Requests' (headers: {'Retry-After': '3', 'X-Retry-In': '2.129390078s', 'Date': 'Tue, 06 Oct 2026 09:23:26 GMT', 'Content-Length': '17'})


OpenEoApiPlainError: [429] Too Many Requests

### 7) Calculate Change between pre and post image

In [ ]:
tresh = 0.3

pre_ndvi = connection.load_result(jobs["PRE"].job_id) 
post_ndvi = connection.load_result(jobs["POST"].job_id)

# 1 = low NDVI (non-vegetated), 0 = vegetated, nodata stays nodata
pre_low  = (pre_ndvi  <= thresh).apply(lambda x: x * 1)
post_low = (post_ndvi <= thresh).apply(lambda x: x * 1)

change_ndvi = (pre_low * 10) + post_low

file_change_ndvi = out_dir / f"change_{event}_NDVI.tif"

# Small extent
# change_ndvi.download(file_change_ndvi, format="GTiff")

# Large extent
job = change_ndvi.create_job(title=f"change_{event}_NDVI", out_format="GTiff")
job.start_and_wait().download_results(out_dir)

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/ecrins/change_Ecrins_2024_mosaics_NDVI.tif
